In [ ]:
import requests
import matplotlib.pyplot as plt
import pandas as pd
import json

# Google Analytics API

In [ ]:
%pip install google-analytics-data pandas

In [ ]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.oauth2 import service_account

PROPERTY_ID = "321460044"

KEY_FILE = r"C:\Users\jlmow\Documents-C Drive\NSS-C Drive\Capstone\sfs-mrktg-76749b6efce7.json"

credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE
)

client = BetaAnalyticsDataClient(credentials=credentials)

print("Connection setup completed.")

In [ ]:
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest
)
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="date"),
        Dimension(name="sessionSourceMedium"),
        Dimension(name="pagePath")
    ],
    metrics=[
        Metric(name="sessions")
    ],
    date_ranges=[
        DateRange(
            start_date="761daysAgo",
            end_date="yesterday"
        )
    ],
 limit=100000
)
response_ga = client.run_report(request)

print("Number of rows returned:", len(response_ga.rows))

In [ ]:
response_ga

# GA4 Data

In [ ]:
data = []

for row in response_ga.rows:
    data.append({
        "date": row.dimension_values[0].value,
        "source_medium": row.dimension_values[1].value,
        "page_path": row.dimension_values[2].value,
        "sessions": row.metric_values[0].value
    })

ga4_df = pd.DataFrame(data)

ga4_df["date"] = pd.to_datetime(
    ga4_df["date"],
    format="%Y%m%d"
)

ga4_df["sessions"] = pd.to_numeric(
    ga4_df["sessions"]

)

ga4_df.sort_values(by="date")

# GA4 - Sum by source_medium

In [ ]:
ga4_df_date = ga4_df.groupby(['source_medium','date']).sum()
ga4_df_date

# GA4 - Sum by Date

In [ ]:
ga4_all = ga4_df.groupby(['date']).sum()
ga4_all
#sum with text concatenates

# HubSpot API

In [ ]:
%pip install requests pandas

In [ ]:
from getpass import getpass

HUBSPOT_TOKEN = getpass("Paste your HubSpot access token: ")

In [ ]:
url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}"
}

all_contacts = []
after = None

while True:

    params = {
    "limit": 100,
    "properties": ",".join([
        "firstname",
        "lastname",
        "email",
        "phone",
        "jobtitle",
        "company",
        "lifecyclestage"
    ])
}

if after is not None:
    params["after"] = after

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

page = response.json()
all_contacts.extend(page.get("results",[]))

print("Number of contacts returned:", len(hubspot_contacts["results"]))

In [ ]:
hs_contacts = []

for contact in hubspot_contacts["results"]:
    properties = contact["properties"]

    hs_contacts.append({
        "contact_id": contact["id"],
        "first_name": properties.get("firstname"),
        "last_name": properties.get("lastname"),
        "email": properties.get("email"),
        "phone": properties.get("phone"),
        "job_title": properties.get("jobtitle"),
        "company_name": properties.get("company"),
        "lifecycle_stage": properties.get("lifecyclestage"),
        "created_at": contact.get("createdAt"),
        "updated_at": contact.get("updatedAt")
    })

hubspot_contacts_df = pd.DataFrame(hs_contacts)

hubspot_contacts_df

#look at UN api and see if you can filter this for parameters

In [ ]:
url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}"
}

property_names = [
    "firstname",
    "lastname",
    "email",
    "phone",
    "jobtitle",
    "company",
    "lifecyclestage",
    "lead_source_1__c"
]

all_contacts = []
after = None

while True:
    params = {
        "limit": 100,
        "properties": ",".join(property_names)
    }

    if after is not None:
        params["after"] = after

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    page = response.json()
    all_contacts.extend(page.get("results", []))

    next_page = page.get("paging", {}).get("next")

    if not next_page:
        break

    after = next_page["after"]

print(f"Downloaded {len(all_contacts):,} contacts.")

In [ ]:
contact_rows = []

for contact in all_contacts:
    properties = contact.get("properties", {})

    contact_rows.append({
        "contact_id": contact.get("id"),
        "first_name": properties.get("firstname"),
        "last_name": properties.get("lastname"),
        "email": properties.get("email"),
        "phone": properties.get("phone"),
        "job_title": properties.get("jobtitle"),
        "company_name": properties.get("company"),
        "lifecycle_stage": properties.get("lifecyclestage"),
        "created_at": contact.get("createdAt"),
        "updated_at": contact.get("updatedAt"),
        "lead_source_1__c": properties.get("source")
    })

hubspot_contacts_df = pd.DataFrame(contact_rows)

hubspot_contacts_df